In [ ]:
# File: force-field.ipynb
# Code: Claude Code
# Review: Ryoichi Ando (ryoichi.ando@zozo.com)
# License: Apache v2.0

# External force fields

A force field adds an acceleration (m/s², like gravity) to every free vertex,
evaluated at the vertex's own position at the start of each step. It comes from
two sources, which can be mixed:

* an exact Python function `def eval(x, y, z, t)` compiled to run on the solver,
  which may call the built-in `noise(...)` and `curl_noise(...)`, and
* a sampled grid over a box, points a given `spacing` apart at `T` instants.

Each source reaches every object, or with `groups=[...]` only the objects put in
those groups with `Object.group(label)`.

The run is then held at a frame, the field is replaced based on where the cloth
is, and the run continues without restarting the solver.

In [ ]:
# import our frontend
from frontend import App
import numpy as np

# make an app
app = App.create("force-field")

# a 1 m square sheet standing in the xy plane
V, F = app.mesh.square(res=32, size=1.0, ex=[1, 0, 0], ey=[0, 1, 0])
app.asset.add.tri("sheet", V, F)

# create a scene with three sheets hanging from their top edges
scene = app.scene.create()
for i in range(3):
    obj = scene.add("sheet").at(1.4 * (i - 1), 1.0, 0)
    obj.pin(obj.grab([0, 1, 0]))
    # the middle sheet gets a group of its own, so a field can target it
    if i == 1:
        obj.group("middle")
    obj.param.set("young-mod", 1000).set("bend", 10.0)

In [ ]:
# THE EXACT FIELD: a swirl about the vertical axis that grows and fades over
# time, for every sheet. Plain Python, compiled for the solver: arithmetic, comparisons,
# if/else, local variables, for loops over range(<number>), abs/min/max and
# math functions are allowed, and every path returns (ax, ay, az).
import math


def eval(x, y, z, t):
    r = math.sqrt(x * x + z * z) + 1e-3
    strength = 6.0 * math.sin(1.5 * t) ** 2
    if r > 2.5:
        return (0.0, 0.0, 0.0)
    return (-z / r * strength, 0.0, x / r * strength)


scene.force_field.script(eval)


# BUILT-IN NOISE, for the middle sheet only: curl noise is a swirling field
# with no sources or sinks. With time=t it changes in place, `frequency` times
# per second, and fades as exp(-decay * t). noise() gives a scalar instead;
# both take octaves (a number from 1 to 8) and a seed.
def gusts(x, y, z, t):
    return curl_noise(2.0 * x, 2.0 * y, 2.0 * z, octaves=3, seed=7,
                      time=t, frequency=1.0, decay=0.2)


scene.force_field.script(gusts, groups=["middle"])

In [ ]:
# THE SAMPLED FIELD: an updraft that rises over the middle sheet, sampled on a
# grid of points every 0.25 m at 4 instants; between samples the solver interpolates.
# The frontend prints how much memory the grid takes.
def updraft(X, Y, Z, t):
    lift = 8.0 * np.exp(-(X**2 + Z**2) / 0.2) * min(t, 1.0)
    return 0 * X, lift, 0 * Z

scene.force_field.sample(
    updraft,
    box_min=(-2.0, -1.0, -2.0),
    box_max=(2.0, 2.0, 2.0),
    spacing=0.25,
    times=[0.0, 0.5, 1.0, 1.5],
)

# compile the scene and report stats
fixed = scene.build().report()
fixed.preview()

In [ ]:
# create a session with the compiled scene
session = app.session.create(fixed)
session.param.set("frames", 120).set("dt", 0.01)
session = session.build()

In [ ]:
# run the first 60 frames and HOLD the solver there. Its process and memory
# stay alive; nothing is saved or restored.
session.run_until_frame(60)
print("held at frame", session.held_frame())

In [ ]:
# steer the field from where the cloth is: pull every sheet back toward the
# center of mass it reached at the held frame. The function is written as
# text so the numbers measured here become constants in it.
if session.held_frame() is not None:
    vert, _ = session.get.vertex(session.held_frame())
    cx, cy, cz = np.asarray(vert)[:, :3].mean(axis=0)
    scene.force_field.clear().script(f'''
def eval(x, y, z, t):
    return (-4.0 * (x - {cx}), 0.0, -4.0 * (z - {cz}))
''')
    session.update_force_field(scene.force_field)

# let the solver run to the end with the new field
session.release()
session.preview()
session.stream()

In [ ]:
session.animate()

In [ ]:
# this is for CI
if app.ci:
    import time
    while session.is_running():
        time.sleep(1)
    assert session.finished()